In [36]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# 1. Load The Splitted Data

In [37]:
# Load the split data
X_train = pd.read_csv('../data/train_data/X_train.csv')
y_train = pd.read_csv('../data/train_data/y_train.csv')
X_test = pd.read_csv('../data/test_data/X_test.csv')
y_test = pd.read_csv('../data/test_data/y_test.csv')

# Display data summaries
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\n--- X_train sample ---")
display(X_train.head())
print("\n--- y_train sample ---")
display(y_train.head())

X_train shape: (6818, 10)
y_train shape: (6818, 1)
X_test shape: (1705, 10)
y_test shape: (1705, 1)

--- X_train sample ---


,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Size,Outlet_Location_Type,Outlet_Type,Outlet_Years
0,9.500,Regular,0.035206,Fruits and Vegetables,171.3448,OUT049,Medium,Tier 1,Supermarket Type1,14
1,18.000,Non-Edible,0.047473,Household,170.5422,OUT045,Unknown,Tier 2,Supermarket Type1,11
2,17.600,Regular,0.076122,Meat,111.7202,OUT046,Small,Tier 1,Supermarket Type1,16
3,8.325,Low Fat,0.029845,Fruits and Vegetables,41.6138,OUT045,Unknown,Tier 2,Supermarket Type1,11
4,12.850,Low Fat,0.137228,Snack Foods,155.5630,OUT046,Small,Tier 1,Supermarket Type1,16



--- y_train sample ---


,Item_Outlet_Sales
0,2386.2272
1,3103.9596
2,1125.2020
3,284.2966
4,4224.5010


# 2. Data Transformation

**Overview & Workflow Sketch**


**Step 1: Manual Ordinal Mapping**
* **Action**: Manually map `Outlet_Size` and `Outlet_Location_Type` to numerical integers (e.g., Unknown: 0, Small: 1, Medium: 2, High: 3).
* **Reasoning**: Manual mapping preserves the natural hierarchy (Small < Medium < High) and treats '**Unknown**' as the baseline (0), which is more logically consistent than alphabetical Label Encoding.

**Step 2: One-Hot Encoding (N-1 Rule)**
* **Action**: Apply **One-Hot Encoding** to nominal variables (`Outlet_Type`, `Item_Fat_Content`, `Item_Type`, `Outlet_Identifier`)
* **Reasoning**: To avoid the **Dummy Variable Trap**, we transform $N$ categories into $N-1$ binary features. This prevents multicollinearity and ensures mathematical stability for regression models.

**Step 3: Feature Scaling**
* **Action**: Standardize numerical features using StandardScaler.
* **Reasoning**: Features have different scales (Visibility: 0.0–0.3 vs. MRP: 30–260). Scaling them to a mean of 0 and standard deviation of 1 prevents features with larger magnitudes from dominating the model's learning process.

**Step 4: Target Transformation (Log Transformation)**
* **Action**: Apply a natural log transformation (`np.log1p`) to the target variable `Item_Outlet_Sales`.
* **Reasoning**: Sales data is typically **right-skewed**. The log transformation "normalizes" the target distribution, which improves the performance, error variance, and stability of the regression algorithm.

## 2.1 Manual Ordial Mapping
Transform `Outlet_Size` and `Outlet_Location_Type` into numerical ranks to preserve their inherent logical hierarchy.

In [38]:
print("--- Before Manual Ordinal Mapping (X_train) ---")
display(X_train[['Outlet_Size', 'Outlet_Location_Type']].head())
print("-" * 40)

# Define the logical hierarchy for Ordinal Variables
# 'Unknown' is assigned 0 as the baseline
size_map = {'Unknown': 0, 'Small': 1, 'Medium': 2, 'High': 3}
location_map = {'Tier 1': 1, 'Tier 2': 2, 'Tier 3': 3}

# Apply the mapping to both Training and Testing sets
for dataset in [X_train, X_test]:
    dataset['Outlet_Size'] = dataset['Outlet_Size'].map(size_map)
    dataset['Outlet_Location_Type'] = dataset['Outlet_Location_Type'].map(location_map)

# Quick check to verify the mapping in X_train
print("--- After Manual Ordinal Mapping (X_train) ---")
display(X_train[['Outlet_Size', 'Outlet_Location_Type']].head())

--- Before Manual Ordinal Mapping (X_train) ---


,Outlet_Size,Outlet_Location_Type
0,Medium,Tier 1
1,Unknown,Tier 2
2,Small,Tier 1
3,Unknown,Tier 2
4,Small,Tier 1


----------------------------------------
--- After Manual Ordinal Mapping (X_train) ---


,Outlet_Size,Outlet_Location_Type
0,2,1
1,0,2
2,1,1
3,0,2
4,1,1


## 2.2 One-Hot Encoding (N-1 Rule)
This step targets the nominal columns `Outlet_Type`, `Item_Fat_Content`, `Item_Type`, and `Outlet_Identifier`. It converts each unique category within these columns into separate binary features ($0$ or $1$). By applying the $N-1$ rule, we remove one redundant column per category to eliminate multicollinearity, ensuring the model's mathematical stability.

In [39]:
# Define nominal columns
nominal_cols = ['Outlet_Type', 'Item_Fat_Content', 'Item_Type', 'Outlet_Identifier']

# Initialize Encoder (N-1 rule & handle unknown categories)
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# Fit and Transform Training set
encoded_train = encoder.fit_transform(X_train[nominal_cols])
encoded_train_df = pd.DataFrame(encoded_train, 
                                columns=encoder.get_feature_names_out(nominal_cols), 
                                index=X_train.index)

# Transform Testing set ONLY (to avoid data leakage)
encoded_test = encoder.transform(X_test[nominal_cols])
encoded_test_df = pd.DataFrame(encoded_test, 
                               columns=encoder.get_feature_names_out(nominal_cols), 
                               index=X_test.index)

# Drop old columns and merge encoded ones
X_train = pd.concat([X_train.drop(columns=nominal_cols), encoded_train_df], axis=1)
X_test = pd.concat([X_test.drop(columns=nominal_cols), encoded_test_df], axis=1)

# Verification
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
display(X_train)

X_train: (6818, 35) | X_test: (1705, 35)


,Item_Weight,Item_Visibility,Item_MRP,Outlet_Size,Outlet_Location_Type,Outlet_Years,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3,Item_Fat_Content_Non-Edible,...,Item_Type_Starchy Foods,Outlet_Identifier_OUT013,Outlet_Identifier_OUT017,Outlet_Identifier_OUT018,Outlet_Identifier_OUT019,Outlet_Identifier_OUT027,Outlet_Identifier_OUT035,Outlet_Identifier_OUT045,Outlet_Identifier_OUT046,Outlet_Identifier_OUT049
0,9.500,0.035206,171.3448,2,1,14,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,18.000,0.047473,170.5422,0,2,11,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,17.600,0.076122,111.7202,1,1,16,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,8.325,0.029845,41.6138,0,2,11,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,12.850,0.137228,155.5630,1,1,16,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6813,9.395,0.286345,139.1838,0,3,15,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6814,15.600,0.117575,75.6670,0,2,6,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6815,17.600,0.018944,237.3590,0,2,11,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6816,20.350,0.054363,117.9466,0,2,6,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3.3 Feature Scaling
This code applies `StandardScaler` to both numerical features (`Item_MRP`, `Item_Weight`, `Item_Visibility`, `Outlet_Years`) and mapped ordinal features (`Outlet_Size`, `Outlet_Location_Type`).

In [40]:
# Initialize the StandardScaler
scaler = StandardScaler()

# List of numerical and ordinal columns to be scaled
num_cols = ['Item_MRP', 'Item_Weight', 'Item_Visibility', 'Outlet_Years', 
            'Outlet_Size', 'Outlet_Location_Type']

# FIT & TRANSFORM the Training set
# This learns the mean and std from X_train only
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# 4. TRANSFORM ONLY the Testing set
# This uses the parameters learned from X_train
X_test[num_cols] = scaler.transform(X_test[num_cols])

# Verification
print("--- After Feature Scaling (Post-Split) ---")
display(X_train[num_cols])

--- After Feature Scaling (Post-Split) ---


,Item_MRP,Item_Weight,Item_Visibility,Outlet_Years,Outlet_Size,Outlet_Location_Type
0,0.470709,-0.733786,-0.709733,-0.136169,0.745846,-1.383482
1,0.457877,1.096432,-0.465070,-0.493521,-1.275290,-0.149659
2,-0.482625,1.010304,0.106311,0.102066,-0.264722,-1.383482
3,-1.603553,-0.986786,-0.816648,-0.493521,-1.275290,-0.149659
4,0.218375,-0.012465,1.325033,0.102066,-0.264722,-1.383482
...,...,...,...,...,...,...
6813,-0.043511,-0.756394,4.299081,-0.017052,-1.275290,1.084165
6814,-1.059078,0.579665,0.933060,-1.089109,-1.275290,-0.149659
6815,1.526207,1.010304,-1.034073,-0.493521,-1.275290,-0.149659
6816,-0.383072,1.602433,-0.327662,-1.089109,-1.275290,-0.149659


## 3.4 Target Transformation (Log Transformation)
Apply the `np.log1p` ($\log(1+x)$) function to the target variable in both `y_train` and `y_test`.

In [41]:
# Apply log1p transformation (log(1+x)) to the target variable
y_train = np.log1p(y_train)
y_test = np.log1p(y_test)

# Verify transformation
print("--- Target Transformation ---")
print(f"y_train Skewness: {y_train.squeeze().skew():.2f}")
print(f"y_test Skewness:  {y_test.squeeze().skew():.2f}")

# Quick look at the transformed target
display(y_train.head())

--- Target Transformation ---
y_train Skewness: -0.89
y_test Skewness:  -0.87


,Item_Outlet_Sales
0,7.777888
1,8.040756
2,7.026606
3,5.653529
4,8.348893


**Data After Standardizing**

In [43]:
display(X_train)

,Item_Weight,Item_Visibility,Item_MRP,Outlet_Size,Outlet_Location_Type,Outlet_Years,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3,Item_Fat_Content_Non-Edible,...,Item_Type_Starchy Foods,Outlet_Identifier_OUT013,Outlet_Identifier_OUT017,Outlet_Identifier_OUT018,Outlet_Identifier_OUT019,Outlet_Identifier_OUT027,Outlet_Identifier_OUT035,Outlet_Identifier_OUT045,Outlet_Identifier_OUT046,Outlet_Identifier_OUT049
0,-0.733786,-0.709733,0.470709,0.745846,-1.383482,-0.136169,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,1.096432,-0.465070,0.457877,-1.275290,-0.149659,-0.493521,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,1.010304,0.106311,-0.482625,-0.264722,-1.383482,0.102066,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,-0.986786,-0.816648,-1.603553,-1.275290,-0.149659,-0.493521,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,-0.012465,1.325033,0.218375,-0.264722,-1.383482,0.102066,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6813,-0.756394,4.299081,-0.043511,-1.275290,1.084165,-0.017052,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6814,0.579665,0.933060,-1.059078,-1.275290,-0.149659,-1.089109,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6815,1.010304,-1.034073,1.526207,-1.275290,-0.149659,-0.493521,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6816,1.602433,-0.327662,-0.383072,-1.275290,-0.149659,-1.089109,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
